# Clase 11 · Escuchar entrevistas con IA: cómo escoger un modelo de audio

Hoy vamos a:

1. Ver qué tareas se pueden hacer con audio.
2. Recorrer **Hugging Face** y aprender a leer la ficha de un modelo.
3. Transcribir una entrevista con un modelo, y luego **cambiar de modelo** y comparar.
4. Obtener marcas de tiempo y guardar la transcripción.
5. Ver cuándo el modelo **inventa**, y qué cuidar antes de usarlo con entrevistas reales.

**Idea central:** el código casi no cambia de un modelo a otro. Lo que cambia, y lo que hay que saber justificar, es **qué modelo** usamos.



## 1. ¿Qué puede hacer la IA con audio?

"Audio" no es una sola tarea. Cada una tiene sus propios modelos:

| Tarea | Qué hace | Ejemplo en investigación |
|---|---|---|
| **Transcripción** (ASR, *Automatic Speech Recognition*) | Convierte voz en texto | Transcribir entrevistas o grupos focales |
| **Traducción** | Convierte voz en un idioma a texto en otro | Entrevistas en kichwa o en inglés |
| **Diarización** | Separa quién habla y cuándo | Distinguir entrevistadora y persona entrevistada |
| **Clasificación de audio** | Pone una etiqueta a un sonido | Voz, música, ruido de calle |
| **Detección de emociones** | Estima el tono emocional de la voz | Muy discutida: el tono no es lo mismo que lo que siente una persona |

Hoy nos concentramos en la **transcripción**.



## 2. Hugging Face: un catálogo de modelos

[Hugging Face](https://huggingface.co/models) es un sitio donde personas, universidades y empresas publican modelos de IA abiertos. Hay cientos de miles. Lo difícil no es encontrar un modelo, sino **escoger uno**.

**Recorrido (hagámoslo juntos en el navegador):**

1. Entren a [huggingface.co/models](https://huggingface.co/models).
2. En la columna izquierda, en *Tasks*, escojan **Automatic Speech Recognition**.
3. En *Languages*, escojan **Spanish**.
4. Ordenen por **Most downloads**.
5. Abran [openai/whisper-small](https://huggingface.co/openai/whisper-small) y [openai/whisper-large-v3-turbo](https://huggingface.co/openai/whisper-large-v3-turbo). Cada modelo tiene una **ficha** (*model card*) que explica qué hace, cómo se entrenó y qué limitaciones tiene.

Mientras leen las fichas, respondan estas **6 preguntas para escoger un modelo**:

| # | Pregunta | Dónde buscar la respuesta |
|---|---|---|
| 1 | ¿Hace la tarea que necesito? | La etiqueta de tarea, arriba de la ficha |
| 2 | ¿Funciona en mi idioma? ¿Con qué calidad? | La lista de idiomas y las tablas de resultados |
| 3 | ¿Qué tamaño tiene? ¿Corre en Colab? | El número de parámetros (por ejemplo, 244M = 244 millones) |
| 4 | ¿Qué licencia tiene? ¿Puedo usarlo en mi investigación? | La etiqueta *License* (MIT, Apache 2.0, CC BY...) |
| 5 | ¿Con qué datos se entrenó y qué limitaciones declara? | Las secciones *Training data* y *Limitations* |
| 6 | ¿Tiene documentación y un ejemplo de uso? | La sección *Usage* o *How to use* |

**Los tamaños de Whisper**

Whisper es una familia de modelos de OpenAI. Todos hacen lo mismo, pero tienen tamaños distintos:

| Modelo | Parámetros | Velocidad | Calidad |
|---|---|---|---|
| `whisper-tiny` | 39 millones | Muy rápido | Baja |
| `whisper-base` | 74 millones | Muy rápido | Baja |
| `whisper-small` | 244 millones | Rápido | Media |
| `whisper-medium` | 769 millones | Lento | Buena |
| `whisper-large-v3` | 1550 millones | Muy lento | La mejor de la familia |
| `whisper-large-v3-turbo` | 809 millones | Rápido | Casi igual que `large-v3` |

Más parámetros suele significar mejor calidad, pero también más memoria y más tiempo.

¿qué dice la ficha sobre los datos de entrenamiento? ¿Sabemos cuántas horas de español latinoamericano escuchó el modelo? ¿Y de español ecuatoriano?

In [ ]:
from huggingface_hub import list_models

# Buscamos modelos especificando la tarea y el idioma directamente en el filtro
modelos = list_models(
    filter=("automatic-speech-recognition", "es"),
    sort="downloads",
    limit=10
)

for modelo in modelos:
    print(modelo.id, modelo.downloads)

## 3. Preparar el entorno

Antes de seguir, activen la **GPU**: menú *Entorno de ejecución* → *Cambiar tipo de entorno de ejecución* → **T4 GPU** → *Guardar*.

La GPU hace que los modelos trabajen mucho más rápido. Sin ella el código funciona igual, pero tarda bastante más.

Colab ya trae las librerías que necesitamos. Solo las importamos:

In [ ]:
import time

import numpy as np
import pandas as pd
import torch
from IPython.display import Audio
from transformers import pipeline

In [ ]:
if torch.cuda.is_available():
    dispositivo = "cuda"
else:
    dispositivo = "cpu"

print("Vamos a usar:", dispositivo)

## 4. Nuestro audio

Usamos una entrevista corta de ejemplo, grabada con el consentimiento de las personas que hablan.

In [ ]:
!wget -q "https://zenodo.org/records/6409263/files/Entrevista1.wav?download=1" -O entrevista_completa.wav

In [ ]:
!ffmpeg -y -loglevel error -i entrevista_completa.wav -ss 60 -t 120 -ac 1 -ar 16000 entrevista.mp3

Escuchemos el audio antes de dárselo al modelo:

In [ ]:
Audio("entrevista.mp3")

**Para usar su propio audio:** ejecuten la celda de abajo y escojan un archivo de su computador (`.mp3`, `.wav` o `.m4a`). Después usen el nombre de su archivo en lugar de `"entrevista.mp3"`.

In [ ]:
from google.colab import files

archivos_subidos = files.upload()

## 5. Primer modelo: `whisper-small`

Para usar un modelo de Hugging Face usamos `pipeline`, igual que en las clases de imágenes y de texto. Le decimos **qué tarea** queremos y **qué modelo** queremos usar.

La primera vez se descarga el modelo (casi 1 GB). Tarda uno o dos minutos.

In [ ]:
transcriptor_pequeno = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    device=dispositivo
)

In [ ]:
inicio = time.time()

resultado_pequeno = transcriptor_pequeno(
    "entrevista.mp3",
    generate_kwargs={"language": "spanish", "task": "transcribe"},
    return_timestamps=True
)

duracion_pequeno = time.time() - inicio

print("Tardó", round(duracion_pequeno), "segundos")
print()
print(resultado_pequeno["text"])

## 6. Cambiar de modelo

Ahora usamos `whisper-large-v3-turbo`.

In [ ]:
transcriptor_turbo = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-large-v3-turbo",
    device=dispositivo
)

In [ ]:
inicio = time.time()

resultado_turbo = transcriptor_turbo(
    "entrevista.mp3",
    generate_kwargs={"language": "spanish", "task": "transcribe"},
    return_timestamps=True
)

duracion_turbo = time.time() - inicio

print("Tardó", round(duracion_turbo), "segundos")
print()
print(resultado_turbo["text"])

Pongamos los dos resultados uno al lado del otro:

In [ ]:
comparacion = pd.DataFrame({
    "modelo": ["whisper-small", "whisper-large-v3-turbo"],
    "segundos": [round(duracion_pequeno), round(duracion_turbo)],
    "palabras": [len(resultado_pequeno["text"].split()), len(resultado_turbo["text"].split())],
    "transcripcion": [resultado_pequeno["text"], resultado_turbo["text"]]
})

comparacion

**Para comentar:**
- ¿Cuál tardó más? ¿Cuál se equivocó menos?
- ¿Las dos transcripciones tienen la misma cantidad de palabras? Si no, ¿dónde está la diferencia?
- Para 30 entrevistas de una hora, ¿cuál escogerían? ¿Y si tuvieran que trabajar sin GPU?

### Tu turno

Busquen en Hugging Face **otro modelo** de transcripción en español (puede ser `openai/whisper-tiny`, `openai/whisper-medium` o uno de la lista de la sección 2) y prueben con el mismo audio.

Pasos:
1. Copien la celda que crea `transcriptor_turbo` y cambien el nombre de la variable y el del modelo.
2. Copien la celda que transcribe y cambien los nombres.
3. Respondan para ese modelo las 6 preguntas de la sección 2.

**Ojo:** algunos modelos de la lista no son de la familia Whisper. Si les da error con `generate_kwargs`, prueben quitando esa línea y revisen la ficha del modelo para ver cómo se usa.

In [ ]:
# escribe aquí

## 7. Marcas de tiempo

Para el análisis cualitativo no basta con el texto: necesitamos saber **en qué momento** se dijo algo, para volver a escucharlo o para citarlo.

Como pedimos `return_timestamps=True`, el resultado trae también el texto dividido en **fragmentos** (`chunks`), cada uno con su inicio y su fin en segundos:

In [ ]:
for fragmento in resultado_turbo["chunks"]:
    print(fragmento["timestamp"], fragmento["text"])

In [ ]:
inicios = []
finales = []
textos = []

for fragmento in resultado_turbo["chunks"]:
    inicios.append(fragmento["timestamp"][0])
    finales.append(fragmento["timestamp"][1])
    textos.append(fragmento["text"])

fragmentos = pd.DataFrame({"inicio": inicios, "fin": finales, "texto": textos})

fragmentos

### Tu turno

Busquen en la tabla los fragmentos que mencionan una palabra que les interese (por ejemplo, "trabajo" o "barrio").

Pista: es igual que en la clase 9, con `str.contains(...)`.

In [ ]:
# escribe aquí

## 8. Guardar la transcripción

Guardamos dos archivos: el texto completo en `.txt` y la tabla con las marcas de tiempo en `.csv`.

In [ ]:
with open("transcripcion.txt", "w", encoding="utf-8") as archivo:
    archivo.write(resultado_turbo["text"])

fragmentos.to_csv("transcripcion_por_fragmentos.csv", index=False)

**Línea por línea**
- `open("transcripcion.txt", "w", encoding="utf-8")` crea un archivo de texto para escribir (`"w"`). `utf-8` permite guardar bien las tildes y la ñ.
- `with ... as archivo:` abre el archivo y lo cierra solo cuando terminamos.
- `fragmentos.to_csv(...)` guarda la tabla, igual que en las clases de pandas.

Los archivos aparecen en la carpeta de la izquierda (ícono de carpeta). Para descargarlos:

In [ ]:
files.download("transcripcion.txt")
files.download("transcripcion_por_fragmentos.csv")

## 9. Cuando el modelo inventa

Whisper aprendió a transcribir escuchando cientos de miles de horas de audio de internet, muchas de ellas de videos con subtítulos. Por eso, cuando el audio tiene silencio o ruido, a veces **escribe cosas que nadie dijo**.
Hagamos la prueba con 10 segundos de puro ruido:

In [ ]:
ruido = np.random.normal(0, 0.01, 16000 * 10).astype("float32")

Audio(ruido, rate=16000)

In [ ]:
resultado_ruido = transcriptor_pequeno(
    {"raw": ruido, "sampling_rate": 16000},
    generate_kwargs={"language": "spanish", "task": "transcribe"}
)

print(resultado_ruido["text"])

## 10. Antes de usarlo con entrevistas reales

Transcribir es también una decisión ética y metodológica. Discutan en grupo:

1. **Consentimiento:** ¿las personas entrevistadas aceptaron que su voz fuera procesada por un modelo de IA? ¿El consentimiento informado lo menciona?
2. **Dónde viajan los datos:** en Colab, el audio se sube a los servidores de Google. ¿Es aceptable para entrevistas sensibles (por ejemplo, con personas en situación de movilidad o víctimas de violencia)? Una alternativa es correr el modelo en un computador propio, sin internet.
3. **Anonimización:** la transcripción incluye nombres, lugares y detalles que permiten identificar a alguien. ¿Cuándo y cómo los van a quitar?
4. **Quién queda fuera:** ¿cómo le va al modelo con acentos de la costa, de la sierra o de la Amazonía? ¿Con personas mayores? ¿Con kichwa o con cambios de idioma en la misma frase?
5. **Qué se pierde:** pausas, silencios, risas, tono, dudas. ¿Qué parte de eso importa para su análisis?

### Tu turno

Escriban en esta celda (doble clic para editar):
- **Para mi proyecto escogería el modelo:**
- **Lo escojo porque (usen al menos 3 de las 6 preguntas):**
- **El mayor riesgo o limitación que veo es:**

## Resumen

- Hugging Face es un catálogo: se filtra por **tarea**, **idioma**, **tamaño** y **licencia**.
- Para escoger un modelo usamos las **6 preguntas**. Las descargas no bastan.
- Con `pipeline`, cambiar de modelo es cambiar **un nombre**. La decisión importante es cuál usar.
- `return_timestamps=True` nos da fragmentos con tiempo, útiles para volver al audio y citar.
- La transcripción automática es un **borrador**: puede equivocarse y también inventar.
- Antes de usar audios reales hay que pensar en el consentimiento, en dónde viajan los datos y en la anonimización.

**Para el taller:** usen las 6 preguntas para escoger el modelo de su proyecto, sea de audio, de imágenes o de documentos. Esa justificación es parte de la presentación del lunes.